In [1]:
import pandas as pd
fact_trades = pd.read_excel ('C:/Users/WestMister/Desktop/Trading analytics/fact_trades.xlsx')

# Исключаем сделки, не соответствующие текущим правилам стратегии или отмеченные как ошибки исполнения.
print('Количество исходных сделок:', len(fact_trades))
fact_trades = fact_trades[fact_trades['note'].isna()].copy()
print('Количество сделок для анализа:', len(fact_trades))
print ('Количество полных дубликатов: ', fact_trades.duplicated().sum())

#Выделим отдельные столбцы, исходя из даты и времени входа в сделку
fact_trades ['open_datetime'] = pd.to_datetime(fact_trades ['open_datetime']).dt.tz_localize(None)
fact_trades ['open_date'] = fact_trades ['open_datetime'].dt.date
fact_trades ['open_date'] = pd.to_datetime (fact_trades ['open_date'])
fact_trades ['open_hour'] = fact_trades ['open_datetime'].dt.hour
fact_trades['is_weekend'] = (fact_trades['open_datetime'].dt.day_name().isin(['Saturday', 'Sunday']))
fact_trades['day_type'] = fact_trades['is_weekend'].map({True: 'weekend',False: 'weekday'})

#Для каждой сделки обозначим риск в USD. В зависимости от даты совершения сделки эта величина будет отличаться.
#Далее приведем результаты всех сделок к одному параметру - риску r. Величину измерения максимально пройденного расстояния ценой тоже переведем из sl в r.
def risk_func(date):
    if date > pd.Timestamp('2026-02-05'):
        return 30
    elif date > pd.Timestamp('2026-01-20'):
        return 15
    else:
        return 10
fact_trades['risk_usd'] = fact_trades['open_date'].apply(risk_func)
fact_trades['result_r'] = (fact_trades['result_$'] / fact_trades['risk_usd']).round(2)
fact_trades['max_move_r'] = (fact_trades['max_move_sl']*0.9).round(1)

fact_trades['pullback_ratio'] = (fact_trades['pullback_depth_m5'] / fact_trades['breakout_depth_m5']).round(1)
fact_trades[['re_entry', 'far_entry']] = fact_trades[['re_entry', 'far_entry']].astype ('boolean')
fact_trades = fact_trades [['ticker', 'open_datetime', 'result_r', 'btc_phase', 'entry_type', 're_entry', 'reentry_opportunity', 'max_move_r', 'r_at_buyer_appearance', 'spike_type', 'spike_size_m5', 
                            'entry_vs_level', 'breakout_depth_m5',  'pullback_depth_m5', 'far_entry', 'pullback_ratio', 'day_type', 'open_hour']]
fact_trades = fact_trades.sort_values('open_datetime').reset_index(drop= True)
fact_trades.info()
fact_trades.to_excel('../data/fact_trades_clean.xlsx', index=False)

Количество исходных сделок: 176
Количество сделок для анализа: 130
Количество полных дубликатов:  0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   ticker                 130 non-null    object        
 1   open_datetime          130 non-null    datetime64[ns]
 2   result_r               130 non-null    float64       
 3   btc_phase              130 non-null    object        
 4   entry_type             130 non-null    object        
 5   re_entry               130 non-null    boolean       
 6   reentry_opportunity    35 non-null     object        
 7   max_move_r             130 non-null    float64       
 8   r_at_buyer_appearance  7 non-null      float64       
 9   spike_type             39 non-null     object        
 10  spike_size_m5          40 non-null     float64       
 11  entry_vs_level         